# Datamine ORIGIN Process Example

This notebook demonstrates how to configure and run the **`origin`** process wrapper in `dmstudio`.

## Process Description

## ORIGIN

**Note** : This is a _superprocess_ and running it may have an effect on other Datamine files in the project.

This process calculates the origin and extent of a rotated model. An optional rotated model prototype file may be created as well as a solid wireframe describing the model limits.

The Rotated Model option allows a block model to be defined, which is not orthogonal to the coordinate axes. The method is described in detail in the Rotated Model User Guide. The orientation of a rotated model is specified by the @**ANGLEn** and @**ROTAXISn** parameters, which are selected to coincide with the geological or structural controls of the orebody. These parameters allow three consecutive rotations around any of the coordinate axes, and are the same as those used in the [CDTRAN](<cdtran.md>) process.

[![](../Images_Commands/origin1.gif)](<javascript:void\(0\);>)

Figure 1: Rotated Model & World Coordinate System.

If the rotated coordinate system can be defined by a single rotation from the world coordinate system, then calculating a suitable origin is relatively straightforward as illustrated in Figure 1. This shows the plan projection of a vertically dipping tabular orebody enclosed by a rectangular outline representing the limits of the rotated model. The origin of the model is the bottom left hand corner, as illustrated. However if two or three rotations are required to align the rotated axes to the orebody, then it is far more difficult to manually calculate the origin and the extent of the model, measured along the rotated axes.

In order to calculate the origin and extent of the rotated model prototype, the process requires a set of X,Y,Z data covering the area to be modeled. If a wireframe of the orebody has been created then the wireframe points file provides a suitable input. Alternatively a set of strings from a sectional interpretation could be used.

Using a wireframe points file or sectional strings would be suitable if the model were required to simply enclose the orebody. If the model is also required to enclose an open pit or underground mine design then a larger model would be required. In this case it would be necessary to provide a set of either points or strings representing the approximate limits of the design.

The optional MARGIN parameter describes the margin around the limits of the data as illustrated in Figure 1. The same MARGIN value is used in each of the three rotated directions. It is recommended that the margin is set to be greater than zero.

The process always calculates and displays the origin and extent of the rotated model. The extent of the rotated model in the X direction is illustrated in Figure 1. In the rotated model prototype the values of XINC (parent cell size) and NX (number of cells in X) must be set such that XINC x NX >= model extent in X; and similarly for Y and Z.

If an OUT file is specified and parameters XINC, YINC, ZINC are defined, then the process will create a rotated model prototype. In the prototype data definition the origin of the rotated system is set to 0, 0, 0, which is suitable for most circumstances.

The optional wireframe file is created around the limits of the prototype model. The purpose of it is simply to provide a visual aid to show the volume within which the actual rotated block model will be created. It is advisable to check the location of the model in this way before proceeding with creating the block model.

### Input Files:

* **in** (Undefined):
  Input file of data covering the volume to be modelled. The data may be a wireframe
  points file, a string file or any file with X, Y and Z fields covering the model volume.
  Required=Yes

### Output Files:

* **out** (Block Model Prototype File):
  Optional output rotated model prototype file.
  Required=No

* **wiretr** (Wireframe Triangle File):
  Optional output wireframe triangle file. The wireframe will be created to enclose the
  limits of the rotated model.
  Required=No

* **wirept** (Wireframe Points File.):
  Optional output wireframe points file. The wireframe will be created to enclose the
  limits of the rotated model.
  Required=No

### Fields:

* **x** (Numeric : IN):
  X coordinate field in IN file.
  Default=XP
  Required=Yes

* **y** (Numeric : IN):
  Y coordinate field in IN file.
  Default=YP
  Required=Yes

* **z** (Numeric : IN):
  Z coordinate field in IN file.
  Default=ZP
  Required=Yes

### Parameters:

* **angle1**:
  First rotation angle clockwise in degrees, around axis ROTAXIS1 . It must lie between
  -360.0 and +360.0. A value of zero indicates no rotation.
  Range=-360,360
  Values=Undefined
  Default=0
  Required=Yes

* **angle2**:
  Second rotation angle clockwise in degrees, around axis ROTAXIS2 . It must lie between
  360.0 and +360.0. A value of zero indicates no rotation.
  Range=-360,360
  Values=Undefined
  Default=0
  Required=No

* **angle3**:
  Third rotation angle clockwise in degrees, around axis ROTAXIS3 . It must lie between
  -360.0 and +360.0. A value of zero indicates no rotation.
  Range=-360,360
  Values=Undefined
  Default=0
  Required=No

* **rotaxis1**:
  Axis around which first rotation angle will occur. 0 for no rotation, 1 for X axis, 2
  for Y axis, 3 for Z axis.
  Range=0,3
  Values=0,1,2,3
  Default=3
  Required=No

* **rotaxis2**:
  Axis around which second rotation angle will occur. 0 for no rotation, 1 for X axis, 2
  for Y axis, 3 for Z axis.
  Range=0,3
  Values=0,1,2,3
  Default=1
  Required=No

* **rotaxis3**:
  Axis around which third rotation angle will occur. 0 for no rotation, 1 for X axis, 2
  for Y axis, 3 for Z axis.
  Range=0,3
  Values=0,1,2,3
  Default=3
  Required=No

* **margin**:
  The margin, in units used in the IN file, to be created around the data volume
  described by the IN file
  Range=Undefined
  Values=Undefined
  Default=10
  Required=No

* **xinc**:
  Parent cell size in X to be created in the output prototype model. This is only
  required if an OUT file has been specified.
  Range=Undefined
  Values=Undefined
  Default=10
  Required=No

* **yinc**:
  Parent cell size in Y to be created in the output prototype model. This is only
  required if an OUT file has been specified.
  Range=Undefined
  Values=Undefined
  Default=10
  Required=No

* **zinc**:
  Parent cell size in Z to be created in the output prototype model. This is only
  required if an OUT file has been specified.
  Range=0.000001,9999999
  Values=Undefined
  Default=10
  Required=No

* **print**:
  Print flag: =0 for minimum output. =1 for runtime information messages.
  Range=0,1
  Values=0,1
  Default=0
  Required=No

In [ ]:
# Step 1: Connect to Datamine and Verify Active Project
import os
import shutil
import glob
import pandas as pd
from dmstudio import dmcommands, dmfiles, initialize, agent

# Connect to running Studio RM instance
dm_cmd = dmcommands.init(version='StudioRM')
dm_fil = dmfiles.init(version='StudioRM')
oScript = initialize.studio('StudioRM')
print(f"Connected to: {oScript.Caption}")

# Verify that the active project matches this folder (case-insensitive) to prevent writing files to the wrong place
active_folder = os.path.normpath(oScript.ActiveProject.Folder).lower()
notebook_folder = os.path.normpath(os.path.dirname(os.path.abspath('__file__'))).lower()

if active_folder != notebook_folder:
    raise RuntimeError(
        f"Active Datamine Project ({active_folder}) does not match this notebook's folder ({notebook_folder}).\n"
        "Please open the 'Project.rmproj' in this folder inside Datamine Studio RM first!"
    )
print("Active project verified successfully.")

## Step 2: Introspect Schema
Always inspect the parameter schema for the process wrapper to see the expected input and output files, fields, and parameters.

In [ ]:
schema = agent.get_command_schema('origin')
print(f"Process: {schema['name']}")
print("Parameters:")
for p in schema['parameters']:
    print(f"  - {p['name']}: default={p['default']}, annotation={p['annotation']}")

## Step 3: Prepare Inputs
We initialize the example project by copying the relevant standard datasets from the Datamine help database locally to this folder using a `t_` prefix. Paths are resolved relatively to ensure portability.

In [ ]:
# Step 3: Initialize example project dataset using relative paths
# Resolve relative path to repository's help database dynamically
repo_root = os.path.abspath(os.path.join(notebook_folder, '..', '..', '..'))
help_db = os.path.join(repo_root, 'datamine_help', 'Database', 'DMTutorials', 'Data', 'VBOP', 'Datamine')

files_to_copy = [
    "_vb_assays.dmx",
    "_vb_collars.dmx",
    "_vb_surveys.dmx",
    "_vb_lithology.dmx",
    "_vb_epar.dmx",
    "_vb_spar.dmx",
    "_vb_vpar.dmx",
    "_vb_mod1.dmx",
    "_vb_SurfacePointsPt.dmx",
    "_vb_SurfaceTriangles.dmx"
]

for filename in files_to_copy:
    src = os.path.join(help_db, filename)
    # Strip _vb_ prefix and prepend t_ for local usage
    local_name = "t_" + filename.replace("_vb_", "")
    dst = os.path.join(notebook_folder, local_name)
    if os.path.exists(src):
        shutil.copy(src, dst)
        print(f"Initialized dataset: {local_name}")
    else:
        print(f"Warning: Source {filename} not found in help database.")

## Step 4: Execute Process
Call the wrapper method with appropriate arguments. Below is the generated template execution call. Required parameters are active, and optional parameters are commented out.

In [ ]:
# Execute origin
print("Running origin...")
dm_cmd.origin(
    in_i='t_assays',  # required
    wiretr_o='t_origin_out',  # required
    wirept_o='t_origin_out',  # required
    x_f='X',  # required
    y_f='Y',  # required
    z_f='Z',  # required
    angles_f=['optional'],  # required
    # out_o='t_origin_out',  # optional
    # rotaxiss_f=['optional'],  # optional
    # margin_p=10,  # optional
    # xinc_p=10,  # optional
    # yinc_p=10,  # optional
    # zinc_p=10,  # optional
    # print_p=0,  # optional
    # arguments='optional',  # optional
    # retrieval='optional',  # optional
)
print("origin execution completed.")

## Step 5: Verify Results
Check that output files exist on disk and read them using pandas to verify the outputs.

In [ ]:
# Step 5: Verify results
output_file = os.path.join(notebook_folder, 't_origin_out.dmx')
if os.path.exists(output_file):
    df = agent.read_datamine(output_file)
    print(f"Output file loaded successfully. Rows: {len(df)}")
    print(df.head())
else:
    print("Output file not found (run Step 4 first)")

## Step 6: Clean up Project Folder
Always clean up generated temporary files to keep the directory clean.

In [ ]:
# Step 6: Clean up temporary files and generated artifacts
# 1. Clean up temporary files matching t_*.*
temp_files = glob.glob(os.path.join(notebook_folder, "t_*.*"))
for f in temp_files:
    try:
        os.remove(f)
        print(f"Removed {os.path.basename(f)}")
    except Exception as e:
        print(f"Failed to remove {os.path.basename(f)}: {e}")

# 2. Clean up dynamic python initialization files (dmdir.py, __init__.py)
extra_files = ['dmdir.py', '__init__.py']
for f in extra_files:
    path = os.path.join(notebook_folder, f)
    if os.path.exists(path):
        try:
            os.remove(path)
            print(f"Removed {f}")
        except Exception as e:
            print(f"Failed to remove {f}: {e}")

# 3. Clean up cache directories (__pycache__)
pycache = os.path.join(notebook_folder, '__pycache__')
if os.path.exists(pycache):
    try:
        shutil.rmtree(pycache)
        print("Removed __pycache__")
    except Exception as e:
        print(f"Failed to remove __pycache__: {e}")